In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap
import geopandas as gpd

# Read Fuel Data

In [ ]:
# read distribution data
distribution_data = pd.read_csv('//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Distribution_2015-2024.csv',low_memory=False)
# filter distribution data
distribution_data = distribution_data[(distribution_data['time']<='2025')]
# create a date column
distribution_data['Date'] = pd.to_datetime(distribution_data['time']).dt.strftime('%Y-%m-%d')
# create a year column
distribution_data['Year'] = pd.to_datetime(distribution_data['time']).dt.year

In [ ]:
# read circuit lengths
circuitlengths = pd.read_excel("//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Q1_2025-_HFRA_Conductor_SumStat_Circuit.xlsx")
# filter for OH only
circuitlengths = circuitlengths[circuitlengths['Conductor Type'].str.contains('OH',na=False)]
# filter for HFRA only
circuitlengths = circuitlengths[circuitlengths['HFRA Tier'].notna()]
# remove blank circuits
circuitlengths = circuitlengths[circuitlengths['Circuit Name'].notna()]
# capitalize circuit names
circuitlengths['Circuit Name'] = circuitlengths['Circuit Name'].str.upper()
circuitlengths_map = circuitlengths.groupby(['Circuit Name','Circuit FLOC'])['Circuit Miles'].sum().reset_index()

In [ ]:
# read forecast distribution data
dist_df_forecast = pd.read_csv('//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingMidput/Dist_Forecasts_2025.csv', low_memory=False)
# make sure Date is date string of Datetime
dist_df_forecast['Date'] = pd.to_datetime(dist_df_forecast['Datetime']).dt.strftime('%Y-%m-%d')

In [ ]:
# for forecast data, groupby Date, Circuit, take the mean of FuelsIndex_max_FPI2
dist_df_forecast_agg = dist_df_forecast.groupby(['Date', 'Circuit'])['FuelsIndex_max_FPI2'].mean().reset_index()
# # for forecast data, groupby Date, Circuit, take the max of FuelsIndex_max_FPI2
# dist_df_forecast_agg = dist_df_forecast.groupby(['Date', 'Circuit'])['FuelsIndex_max_FPI2'].max().reset_index()
# merge with circuit lengths
dist_df_forecast_agg = dist_df_forecast_agg.merge(circuitlengths_map, left_on='Circuit', right_on='Circuit Name', how='left')
# locate rows where Circuit Miles is NaN and Circuit contains ambercrest and change value to 32.7754002840092
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('AMBERCREST', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['AMBERCREST',32.7754002840092]
# locate rows where Circuit Miles is NaN and Circuit contains aventura and change value to 15.7032769637841
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('AVENTURA', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['AVENTURA',15.7032769637841]
# locate rows where Circuit Miles is NaN and Circuit contains birchim and change value to 46.34484383
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('BIRCHIM', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['BIRCHIM',46.34484383]
# locate rows where Circuit Miles is NaN and Circuit contains cafe and change value to 27.6402669615555
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('CAFE', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['CAFE',27.6402669615555]
# locate rows where Circuit Miles is NaN and Circuit contains canebrake and change value to 66.16859376
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('CANEBRAKE', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['CANEBRAKE',66.16859376]
# locate rows where Circuit Miles is NaN and Circuit contains casals and change value to 0.446999553409736
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('CASALS', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['CASALS',0.446999553409736]
# locate rows where Circuit Miles is NaN and Circuit contains coachella and change value to 6.590021052
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('COACHELLA', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['COACHELLA',6.590021052]
# locate rows where circuit miles is NaN and circuit contains energy and change value to 32.37798311
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('ENERGY', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['ENERGY',32.37798311]
# locate rows where circuit miles is NaN and circuit contains fingal and change value to 36.6548594697613
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('FINGAL', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['FINGAL',36.6548594697613]
# locate rows where circuit miles is NaN and circuit contains guitar and change value to 42.2491638220717
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('GUITAR', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['GUITAR',42.2491638220717]
# locate rows where circuit miles is NaN and circuit contains huckleberry and change value to 25.4840533042463
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('HUCKLEBERRY', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['HUCKLEBERRY',25.4840533042463]
# locate rows where circuit miles is NaN and circuit contains Kinsey and change value to 38.2093922
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('KINSEY', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['KINSEY',38.2093922]
# locate rows where circuit miles is NaN and circuit contains Penstock and change value to 50.90351311
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('PENSTOCK', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['PENSTOCK',50.90351311]
# locate rows where circuit miles is NaN and circuit contains Shovel and change value to 47.51064986
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('SHOVEL', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['SHOVEL',47.51064986]
# locate rows where circuit miles is NaN and circuit contains Tejon and change value to 57.8330462
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('TEJON', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['TEJON',57.8330462]
# locate rows where circuit miles is NaN and circuit contains Toll and change value to 8.228283341
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('TOLL', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['TOLL',8.228283341]
# locate rows where circuit miles is NaN and circuit contains Sutt and change value to 16.8635623
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('SUTT', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['SUTT',16.8635623]
# locate rows where circuit miles is NaN and circuit contains POPPET FLATS and change value to 31.30335437
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('POPPET FLATS', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['POPPET FLATS',31.30335437]
# sort values by Date and Circuit Name and FuelsIndex_max_FPI2 descending
dist_df_forecast_agg = dist_df_forecast_agg.sort_values(by=['Date', 'Circuit Name', 'FuelsIndex_max_FPI2'], ascending=[True, True, False])
# # drop duplicate Date and Circuit Name, keep first one
dist_df_forecast_agg = dist_df_forecast_agg.drop_duplicates(subset=['Date', 'Circuit Name'], keep='first')

# Monthly Forecast

In [ ]:
# read ignition data
ignitions = pd.read_excel("//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Ignition_Safari_Connection.xlsx")
# rename columns 
ignitions = ignitions.rename(columns={'Date/Time':'Date','Circuit Name':'Circuit_Name','Circuit ID':'Circuit_ID','Event ID':'Event_ID'})
# get year column
ignitions['Year'] = pd.to_datetime(ignitions['Date']).dt.year
# get month column
ignitions['Month'] = pd.to_datetime(ignitions['Date']).dt.month
# keep only SCE involved ignitions
ignitions = ignitions[ignitions['SCE Involved'].str.contains('yes|inconclusive',case=False,na=False)]
# keep only reportable ignitions
ignitions = ignitions[ignitions['CPUC Reportable']==1]
# for Status, ignore the ones that say open
ignitions = ignitions[~ignitions['Status'].str.contains('open', case=False, na=False)]
# only keep ones where Status says complete
ignitions = ignitions[ignitions['Status'].str.contains('complete', case=False, na=False)]
# keep only HFRA ignitions
ignitions = ignitions[ignitions['HFRA'].str.contains('T2|T3',na=False)]
# loc row where circuit name has 'banducci' and change Circuit_ID to ET-02156
ignitions.loc[ignitions['Circuit_Name'].str.contains('banducci', case=False, na=False), 'Circuit_ID'] = 'ET-02156'
# change date format
ignitions['Date'] = pd.to_datetime(ignitions['Date']).dt.strftime('%Y-%m-%d')
# from 2015 onward
ignitions = ignitions[ignitions['Year'] >= 2015]
# # before August 1 2025
# ignitions = ignitions[ignitions['Date'] < '2025-08-01']
# group by Year and get count of ignitions
ignitions = ignitions.groupby(['Year','Month']).agg({'Event_ID': 'count'}).reset_index().rename(columns={'Event_ID': 'Actual Ignitions'})
# create a dataframe with all months from 2015 to 2025
all_months = pd.DataFrame({'Year': np.repeat(range(2015, 2026), 12), 'Month': list(range(1, 13)) * 11})
# merge with ignitions to get all months
ignitions = all_months.merge(ignitions, on=['Year', 'Month'], how='left')
# fill na with 0
ignitions['Actual Ignitions'] = ignitions['Actual Ignitions'].fillna(0).astype(int)
ignitions

## Option 2/3: Weighted Ignitions over Weighted MileDays, Cumulative